# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Employee_HR_Analytics

**Input file:** `04_Employee_HR.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("04_Employee_HR.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["salary"].agg(["count","mean","min","max"]))

## Q2. High-value records

Identify records above the 75th percentile of salary.

In [ ]:
t=    employee_id employee_name  ... joining_date employment_status
0         E0001     Nisha Roy  ...   2016-05-01            Active
1         E0002   Priya Gupta  ...   2024-04-06            Active
2         E0003     Nisha Roy  ...   2017-11-17            Active
3         E0004  Aakash Kumar  ...   2019-08-03          Inactive
4         E0005   Karan Verma  ...   2012-01-20            Active
..          ...           ...  ...          ...               ...
317       E0318    Neha Singh  ...   2014-05-24            Active
318       E0319    Neha Singh  ...   2012-10-15            Active
319       E0320   Priya Gupta  ...   2012-10-30            Active
320       E0012   Karan Verma  ...   2016-05-31            Active
321       E0034   Karan Verma  ...   2018-02-03            Active

[322 rows x 10 columns]
threshold=df["salary"].quantile(.75)
print(df[df["salary"]>threshold].sort_values("salary",ascending=False).head(20))

## Q3. Category aggregation

Calculate record count, total salary, and average experience_years for each employee_id.

In [ ]:
print(df.groupby("employee_id",dropna=False).agg(count=("salary","count"),total=("salary","sum"),avg=("experience_years","mean")).sort_values("total",ascending=False))

## Q4. Two-dimensional aggregation

Analyze salary by employee_id and employee_name; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["employee_id","employee_name"],dropna=False).agg(count=("salary","count"),total=("salary","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q5. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["salary"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q6. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="employee_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q7. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("employee_id"); print("Before:",len(df),"After:",len(x))

## Q8. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q9. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["employee_id"]=x["employee_id"].fillna("Unknown"); print(x["employee_id"].value_counts(dropna=False))

## Q10. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["salary"]=x["salary"].fillna(x["salary"].median()); print(x["salary"].isna().sum())

## Q11. String search

Find records where employee_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["employee_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q12. Sorted analysis

Sort the dataset by salary descending and experience_years ascending.

In [ ]:
print(df.sort_values(["salary","experience_years"],ascending=[False,True]).head(25))

## Q13. Conditional segmentation

Create a new segment based on salary using its median as the threshold.

In [ ]:
x=df.copy(); med=x["salary"].median(); x["segment"]=np.where(x["salary"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q14. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("employee_id")["salary"].transform("mean"); print(x[["employee_id","salary","group_avg"]].head(25))

## Q15. Group share

Calculate each employee_id's share of total salary.

In [ ]:
g=df.groupby("employee_id",dropna=False)["salary"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q16. Pivot report

Create a pivot table showing average salary by employee_id and employee_name.

In [ ]:
print(pd.pivot_table(df,index="employee_id",columns="employee_name",values="salary",aggfunc="mean"))

## Q17. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("employee_id")[["salary","experience_years"]].mean().reset_index(); print(w.melt(id_vars="employee_id",var_name="metric",value_name="value"))

## Q18. Validation rule

Count records where salary is negative or missing.

In [ ]:
print("Invalid:",df["salary"].isna().sum()+(df["salary"]<0).sum())

## Q19. Outlier detection

Use IQR to identify high outliers in salary.

In [ ]:
q1=df["salary"].quantile(.25); q3=df["salary"].quantile(.75); iqr=q3-q1; print(df[df["salary"]>q3+1.5*iqr][["salary"]].head(30))

## Q20. Top group

Find the category in employee_id with the highest total salary.

In [ ]:
print(df.groupby("employee_id")["salary"].sum().nlargest(1))

## Q21. Bottom group

Find the three categories in employee_id with the lowest average salary.

In [ ]:
print(df.groupby("employee_id")["salary"].mean().nsmallest(3))

## Q22. Multiple aggregations

Produce min, max, mean and median of salary for each employee_id.

In [ ]:
print(df.groupby("employee_id")["salary"].agg(["min","max","mean","median"]))

## Q23. Apply rule

Use apply to classify salary relative to its median.

In [ ]:
x=df.copy(); med=x["salary"].median(); x["classification"]=x["salary"].apply(lambda v:"High" if pd.notna(v) and v>=med else "Low"); print(x["classification"].value_counts())

## Q24. Map categories

Create a numeric code for the most frequent categories in the first categorical field.

In [ ]:
codes={{v:i for i,v in enumerate(df["employee_id"].dropna().unique())}}; x=df.copy(); x["employee_id_code"]=x["employee_id"].map(codes); print(x[["employee_id","employee_id_code"]].head(20))

## Q25. ETL filtering

Build an ETL subset containing non-null salary, non-null employee_id, and values above the median salary.

In [ ]:
m=df["salary"].median(); x=df[df["salary"].notna()&df["employee_id"].notna()&(df["salary"]>m)].copy(); print(x.shape); print(x.head())

## Q26. Chunk processing

Read the input CSV in chunks of 50 rows and calculate the main numeric total.

In [ ]:
total=0
count=0
for chunk in pd.read_csv("04_Employee_HR.csv",chunksize=50):
    total += chunk["salary"].sum()
    count += len(chunk)
print("Rows processed:",count)
print("Total:",total)

## Q27. Final report

Create a final employee_id-level report with count, total salary, average experience_years, and rank by total salary.

In [ ]:
r=df.groupby("employee_id",dropna=False).agg(records=("salary","count"),total=("salary","sum"),average=("experience_years","mean")).sort_values("total",ascending=False); r["rank"]=r["total"].rank(method="dense",ascending=False).astype(int); print(r)

## End of Project

The notebook is ready to execute with the supplied input CSV.